# D4-1: a learned direct critic against a co-state critic (validation-only; the test family is never generated)

Follow-up to D4-3 (`docs/research-notes/2026-09-30-d4-3-varying-goal.md`), which reopened D4-1 in `m4` and `m64`. The full design, the prices, the frozen rules, the exit classes and the list of changes made before this run are in `docs/plans/d4-1-plan.md`; read that first. **This notebook is a design study, not a confirmatory gate.** It uses D4-3's tuning (seed 3003) and validation (seed 1001) instances of the cells `m4` and `m64`, each with its own random unit goal, and a *train* family (seed 4004, 30 instances) for the critics. **The test family (seed 2002) is never generated or read.** It runs on CPU in NumPy.

**What it does.** For each cell it trains, for five paired seeds, a co-state estimator `ŵ(t_b, c)` on the co-state label (and a control on permuted labels) and four value heads on the gain label: `direct` (P0 inputs only), `costate_critic` (P0 inputs plus three features from the trained estimator), `costate_randomised` (the same with the permuted-label estimator) and `teacher_feature` (features from the exact co-state; a privileged diagnostic, reported at the lookup price only). The head width is chosen on the tuning family at the lookup price (ranking quality) with the direct arm as the yardstick; `θ` is chosen per (cell, arm, seed) on the tuning family at both prices; everything is then frozen and hashed. Only then is the validation family generated and every arm run. Compute to reach 10 %, 3 % and 1 % of the initial-to-finest gap is priced by log-log interpolation (primary), with semilog and staircase as sensitivities, at the **real price** and at the **hypothetical lookup price**.

**Frozen rules** (plan §7; "beats" is the 95 % interval of the two-stage bootstrap of the mean log compute ratio below 0 at two adjacent targets): R0, RC1, RC2, RC3, RT-L, the estimator floor, and the exit class per cell.

**Limits.** No H2 statement about real data. The co-state is linear in the goal; a generic small MLP may not learn it (see the plan §9 and §10); the real-price gate is expected to fail; the lookup price is hypothetical; one system per cell; twenty validation instances and five seeds; the validation instances were read in D4-3.

In [ ]:
# ================================================================
# 0. Package (CPU is enough)
# ================================================================
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/romusking/para_001'
REPO_REF = 'main'  # branch, tag or commit SHA
REPO_DIR = '/content/para_001' if IN_COLAB else str(Path.cwd().resolve().parents[1])

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', '--quiet', REPO_URL, REPO_DIR], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--quiet', '--all'], check=False)
    for candidate in (f'origin/{REPO_REF}', REPO_REF):
        resolved = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', '--verify', '--quiet', candidate + '^{commit}'],
                                  capture_output=True, text=True)
        if resolved.returncode == 0:
            subprocess.run(['git', '-C', REPO_DIR, 'checkout', '--quiet', '--detach', resolved.stdout.strip()], check=True)
            break
    else:
        raise RuntimeError(f'Cannot resolve {REPO_REF!r}.')
REPO_COMMIT = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], text=True).strip()
REPO_DIRTY = bool(subprocess.check_output(['git', '-C', REPO_DIR, 'status', '--porcelain'], text=True).strip())
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
try:
    from adjointrwm.domains import critics
    critics.train_critic  # noqa: B018
except (ImportError, AttributeError) as error:
    raise RuntimeError(f'{REPO_REF!r} predates the D4-1 additions; set REPO_REF to a commit that has them.') from error

if IN_COLAB:
    from google.colab import drive

    drive.mount('/content/drive', force_remount=False)
    RUN_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/AdjointRWM_Production/runs')
else:
    RUN_ROOT = Path.home() / 'adjointrwm_runs'
print('adjointrwm at', REPO_COMMIT, '(uncommitted changes!)' if REPO_DIRTY else '', '| runs ->', RUN_ROOT)

In [ ]:
# ================================================================
# 1. Frozen configuration, the immutable run directory and the train and tuning instances
# ================================================================
import dataclasses
import datetime
import json
import platform
import time
from dataclasses import asdict, dataclass

import numpy as np
import pandas as pd

from adjointrwm.domains import AdaptiveTimeSteppingDomain, critics as cr, highdim as hd
from adjointrwm.io import atomic_write_json, atomic_write_text, sha256_json


@dataclass(frozen=True)
class D41Config:
    cells: tuple = ('m4', 'm64')                   # the two D4-3 candidate cells
    train_instances: int = 30                      # critic training (seed 4004, goals from their own stream)
    tuning_instances: int = 10                     # theta and width (seed 3003)
    validation_instances: int = 20                 # reported (seed 1001)
    # The test family (seed 2002) is deliberately never generated in this notebook.
    seeds: tuple = (0, 1, 2, 3, 4)
    collection_theta: float = 0.85                 # marking fraction of the collection policies (as D4-2)
    per_state: int = 64
    goals_per_state: int = 4
    max_rows: int = 60_000
    estimator_hidden: int = 32
    estimator_max_epochs: int = 200
    estimator_patience: int = 20
    estimator_batch: int = 256
    estimator_floor: float = 0.5                   # held-out relative error against the exact co-state
    head_max_epochs: int = 40
    head_patience: int = 6
    widths: tuple = (8, 16, 32)
    capacity_probe_width: int = 64
    width_theta: float = 0.95
    width_tolerance: float = 0.02
    thetas: tuple = (0.5, 0.7, 0.85, 0.95, 0.98, 0.995)
    reference_arms: tuple = ('cheap', 'cheap_adjoint', 'adjoint')
    gap_fractions: tuple = (0.1, 0.03, 0.01)
    cap_in_finest_grids: float = 2.0
    max_passes: int = 40
    primary_method: str = 'loglog'
    methods: tuple = ('loglog', 'semilog', 'staircase')
    bootstrap_resamples: int = 10000
    equivalence_margin: float = float(np.log(1.05))
    seeds_needed_for_estimator_floor: int = 4


CFG = D41Config()
CONFIG_DICT = asdict(CFG)
CELL_TABLE = {name: asdict(hd.CELLS[name]) for name in CFG.cells}
CONFIG_HASH = sha256_json({'config': CONFIG_DICT, 'cells': CELL_TABLE, 'family_seeds': hd.FAMILY_SEEDS, 'goal_stream_offset': hd.GOAL_STREAM_OFFSET})
RUN_ID = 'd4_1_learned_critics_' + datetime.datetime.now(datetime.UTC).strftime('%Y%m%dT%H%M%SZ')
RUN_DIR = RUN_ROOT / RUN_ID
if RUN_DIR.exists():
    raise RuntimeError(f'{RUN_DIR} exists; run directories are immutable.')
PATHS = {k: RUN_DIR / k for k in ('config', 'artifacts', 'figures', 'reports', 'tests')}
for path in PATHS.values():
    path.mkdir(parents=True, exist_ok=True)

DOMAIN = AdaptiveTimeSteppingDomain(objective_kind='bound')
TRAIN_INSTANCES = {cell: hd.varying_goal_instances(cell, 'train', CFG.train_instances) for cell in CFG.cells}
TUNING_INSTANCES = {cell: hd.varying_goal_instances(cell, 'tuning', CFG.tuning_instances) for cell in CFG.cells}
TARGETS = list(hd.TARGETS)


def targets_fn(domain, instance):
    return hd.gap_targets(domain, instance, CFG.gap_fractions)


def cap_of(instance):
    return CFG.cap_in_finest_grids * instance.n_fine


def instance_table(instances_by_cell, family):
    return pd.DataFrame([{'cell': cell, 'family': family, 'name': i.name, 'goal': list(i.goal), 'y0': list(i.y0), 'n_fine': i.n_fine,
                          'pulses_json': json.dumps([asdict(p) for p in i.pulses], separators=(',', ':'))}
                         for cell, insts in instances_by_cell.items() for i in insts])


def manifest_of(instances_by_cell, family, count):
    return {cell: {'goal_seed': hd.FAMILY_SEEDS[family] + hd.GOAL_STREAM_OFFSET, 'family_seed': hd.FAMILY_SEEDS[family], 'count': count,
                   'names': [insts[0].name, insts[-1].name], 'sha256': sha256_json([asdict(i) for i in insts])}
            for cell, insts in instances_by_cell.items()}


instance_table(TRAIN_INSTANCES, 'train').to_parquet(PATHS['config'] / 'instances_train.parquet', index=False)
instance_table(TUNING_INSTANCES, 'tuning').to_parquet(PATHS['config'] / 'instances_tuning.parquet', index=False)
TRAIN_MANIFEST = manifest_of(TRAIN_INSTANCES, 'train', CFG.train_instances)
TUNING_MANIFEST = manifest_of(TUNING_INSTANCES, 'tuning', CFG.tuning_instances)
atomic_write_json(PATHS['config'] / 'instances_manifest_train.json', {**TRAIN_MANIFEST, 'sha256': sha256_json(TRAIN_MANIFEST)})
atomic_write_json(PATHS['config'] / 'instances_manifest_tuning.json', {**TUNING_MANIFEST, 'sha256': sha256_json(TUNING_MANIFEST)})
atomic_write_json(PATHS['config'] / 'run_config.json', {
    'run_id': RUN_ID, 'config': CONFIG_DICT, 'cells': CELL_TABLE, 'family_seeds': hd.FAMILY_SEEDS, 'goal_stream_offset': hd.GOAL_STREAM_OFFSET,
    'config_sha256': CONFIG_HASH, 'repo_commit': REPO_COMMIT, 'repo_dirty': REPO_DIRTY, 'notebook': 'notebooks/04-domains/d4_1_learned_critics.ipynb',
    'plan': 'docs/plans/d4-1-plan.md', 'test_family_read': False,
})
atomic_write_json(PATHS['config'] / 'environment.json', {
    'python': sys.version, 'platform': platform.platform(), 'numpy': np.__version__, 'pandas': pd.__version__,
    'hardware': 'CPU (FLOPs are counted, not measured)',
})
print('Run:', RUN_ID, '| config', CONFIG_HASH[:12], '|', len(CFG.cells), 'cells,', CFG.train_instances, 'train,', CFG.tuning_instances, 'tuning instances each')

In [ ]:
# ================================================================
# 2. Correctness and ledger checks on train and tuning instances (before anything is trained)
# ================================================================
from adjointrwm.domains.linear_ode import ODEState, marking_policy, uniform_pass_policy

rows = []
for cell in CFG.cells:
    inst = TUNING_INSTANCES[cell][0]
    m, pulses = inst.m, len(inst.pulses)
    nodes = tuple(range(0, inst.n_fine + 1, 4))
    state = ODEState(inst, nodes, DOMAIN.stepper(inst).solve(nodes))
    # the propagator reproduces the tabulated co-state of this instance's goal, and the discrete sweep for many goals equals domain.costate
    propagator_gap = float(np.abs(cr.propagator(inst) @ np.asarray(inst.goal) - hd.costate_table(inst)).max())
    other = TUNING_INSTANCES[cell][1]
    many = cr.discrete_costates(DOMAIN, inst, state.nodes, np.array([inst.goal, other.goal]))
    sweep_gap = float(max(np.abs(many[0] - DOMAIN.costate(state)).max(),
                          np.abs(many[1] - DOMAIN.costate(ODEState(dataclasses.replace(inst, goal=other.goal), state.nodes, state.y))).max()))
    # the exact-co-state features of the teacher equal the terms of the cheap_adjoint score
    d, e = hd.forcing_discrepancy(inst, state), hd.difference_estimate(inst, state)
    w = hd.costate_table(inst)[np.asarray(state.nodes[1:])]
    feats = cr.costate_features(w, d, e)
    teacher_gap = float(np.abs(10.0 ** feats[:, 0] + 10.0 ** feats[:, 1] - 2 * cr.LABEL_FLOOR
                               - (np.abs(np.einsum('jm,jm->j', w, d)) + np.abs(np.einsum('jm,jm->j', w, e)))).max())
    # a label of the objective's own summand
    tau = DOMAIN.exact_local_errors(state, inst)
    label_gap = float(np.abs(10.0 ** cr.gain_labels(many[0], tau) - cr.LABEL_FLOOR - np.abs(DOMAIN.weighted_local_errors(state, inst))).max())
    counted_step = 2 * (2 * m * m) + m + m + m + 2 * pulses * (2 * m + 8)
    train_goals = {tuple(i.goal) for i in TRAIN_INSTANCES[cell]}
    rows.append({'cell': cell, 'm': m, 'flops_step_formula': hd.flops_step(inst), 'flops_step_counted': counted_step,
                 'propagator_vs_table_max_abs': propagator_gap, 'discrete_sweep_vs_domain_max_abs': sweep_gap,
                 'teacher_features_vs_cheap_adjoint_terms_max_abs': teacher_gap, 'gain_label_vs_objective_summand_max_abs': label_gap,
                 'train_goals_disjoint_from_tuning': not (train_goals & {tuple(i.goal) for i in TUNING_INSTANCES[cell]}),
                 'train_names_disjoint_from_tuning': not ({i.name for i in TRAIN_INSTANCES[cell]} & {i.name for i in TUNING_INSTANCES[cell]})})
checks = pd.DataFrame(rows)
CORRECTNESS = {
    'flop_formula_matches_counted_code_path': bool((checks['flops_step_formula'] == checks['flops_step_counted']).all()),
    'propagator_equals_costate_table': bool((checks['propagator_vs_table_max_abs'] < 1e-9).all()),
    'discrete_sweep_equals_domain_costate': bool((checks['discrete_sweep_vs_domain_max_abs'] < 1e-10).all()),
    'teacher_features_are_the_cheap_adjoint_terms': bool((checks['teacher_features_vs_cheap_adjoint_terms_max_abs'] < 1e-12).all()),
    'gain_label_is_the_objectives_own_summand': bool((checks['gain_label_vs_objective_summand_max_abs'] < 1e-12).all()),
    'train_goals_disjoint_from_tuning': bool(checks['train_goals_disjoint_from_tuning'].all()),
    'train_names_disjoint_from_tuning': bool(checks['train_names_disjoint_from_tuning'].all()),
    'max_propagator_gap': float(checks['propagator_vs_table_max_abs'].max()),
    'max_sweep_gap': float(checks['discrete_sweep_vs_domain_max_abs'].max()),
    'test_family_refused': False,
}
try:
    hd.varying_goal_instances('m4', 'test', 1)
except ValueError:
    CORRECTNESS['test_family_refused'] = True
CORRECTNESS['passed'] = bool(all(v for k, v in CORRECTNESS.items() if isinstance(v, bool)))
atomic_write_json(PATHS['tests'] / 'correctness.json', CORRECTNESS)
checks.to_csv(PATHS['tests'] / 'ledger_and_label_checks.csv', index=False)
print(json.dumps(CORRECTNESS, indent=2))
if not CORRECTNESS['passed']:
    raise RuntimeError('Correctness failed: nothing below may be trained. See tests/.')

In [ ]:
# ================================================================
# 3. Training states, the width of the heads and the capacity probe (train and tuning instances only)
# ================================================================
COLLECTION = [uniform_pass_policy(), marking_policy('residual', CFG.collection_theta), marking_policy('adjoint', CFG.collection_theta)]
STATES, WIDTHS, WIDTH_ROWS = {}, {}, []
started = time.perf_counter()
for cell in CFG.cells:
    train, tuning = TRAIN_INSTANCES[cell], TUNING_INSTANCES[cell]
    m = train[0].m
    STATES[cell] = cr.collect_states(DOMAIN, train, COLLECTION, CFG.max_passes, CFG.per_state, seed=0)
    print(f'{cell}: {len(STATES[cell])} training states, {sum(len(s.intervals) for s in STATES[cell])} sampled intervals, {time.perf_counter() - started:.0f}s elapsed', flush=True)
    rows0 = cr.build_rows(DOMAIN, STATES[cell], train, seed=0, goals_per_state=CFG.goals_per_state, max_rows=CFG.max_rows)
    mask0 = cr.split_by_group(rows0.group, 0)
    est0 = cr.train_estimator(rows0.take(~mask0), rows0.take(mask0), CFG.estimator_hidden, 0, max_epochs=CFG.estimator_max_epochs,
                              patience=CFG.estimator_patience, batch_size=CFG.estimator_batch)
    n_direct, n_costate = hd.FEATURE_DIM + 3 * m, hd.FEATURE_DIM + 3 * m + cr.COSTATE_FEATURES

    def probe(width):
        h_direct = cr.match_direct_hidden(n_direct, n_costate, width, est0.dims)
        direct = cr.train_critic('direct', rows0, h_direct, 0, None, CFG.head_max_epochs, CFG.head_patience)
        costate = cr.train_critic('costate_critic', rows0, width, 0, est0, CFG.head_max_epochs, CFG.head_patience)
        cost = {name: cr.tune_critic_theta(DOMAIN, tuning, critic, (CFG.width_theta,), 'lookup', targets_fn, cap_of, CFG.max_passes, CFG.primary_method)[CFG.width_theta]
                for name, critic in (('direct', direct), ('costate_critic', costate))}
        WIDTH_ROWS.append({'cell': cell, 'width': width, 'direct_width': h_direct, 'direct_lookup_compute': cost['direct'], 'costate_lookup_compute': cost['costate_critic'],
                           'direct_val_rmse_log10': direct.report['val_rmse_log10'], 'costate_val_rmse_log10': costate.report['val_rmse_log10'],
                           'direct_spearman': direct.report['val_spearman_within_state'], 'costate_spearman': costate.report['val_spearman_within_state']})
        return h_direct, cost['direct']

    direct_cost = {w: probe(w)[1] for w in CFG.widths}
    best = min(direct_cost.values())
    chosen = min(w for w, c in direct_cost.items() if c <= (1.0 + CFG.width_tolerance) * best)
    best_width = min(direct_cost, key=direct_cost.get)
    saturated, raised = True, False
    if best_width == max(CFG.widths):
        raised_cost = probe(CFG.capacity_probe_width)[1]
        if raised_cost <= (1.0 - CFG.width_tolerance) * best:
            chosen, raised, saturated = CFG.capacity_probe_width, True, False
    WIDTHS[cell] = {'width': chosen, 'direct_width': cr.match_direct_hidden(n_direct, n_costate, chosen, est0.dims), 'direct_saturated': saturated,
                    'envelope_raised_by_capacity_probe': raised, 'direct_lookup_compute_by_width': {str(k): v for k, v in direct_cost.items()}}
    print(f'{cell}: head width {chosen} (direct width {WIDTHS[cell]["direct_width"]}), direct saturated: {saturated}, {time.perf_counter() - started:.0f}s elapsed', flush=True)
WIDTH_TABLE = pd.DataFrame(WIDTH_ROWS)
WIDTH_TABLE.to_csv(PATHS['artifacts'] / 'width_selection.csv', index=False)
print(WIDTH_TABLE.to_string(index=False))

In [ ]:
# ================================================================
# 4. Five paired seeds: estimators, then heads (train and tuning instances only)
# ================================================================
CRITICS, TRAIN_ROWS = {}, []
started = time.perf_counter()
for cell in CFG.cells:
    train, m = TRAIN_INSTANCES[cell], TRAIN_INSTANCES[cell][0].m
    h, h_direct = WIDTHS[cell]['width'], WIDTHS[cell]['direct_width']
    for seed in CFG.seeds:
        rows_s = cr.build_rows(DOMAIN, STATES[cell], train, seed=seed, goals_per_state=CFG.goals_per_state, max_rows=CFG.max_rows)
        mask = cr.split_by_group(rows_s.group, seed)
        tr, va = rows_s.take(~mask), rows_s.take(mask)
        estimators = {permute: cr.train_estimator(tr, va, CFG.estimator_hidden, seed, permute=permute, max_epochs=CFG.estimator_max_epochs,
                                                  patience=CFG.estimator_patience, batch_size=CFG.estimator_batch) for permute in (False, True)}
        spec = {'direct': (h_direct, None), 'costate_critic': (h, estimators[False]), 'costate_randomised': (h, estimators[True]), 'teacher_feature': (h, None)}
        for arm, (width, estimator) in spec.items():
            critic = cr.train_critic(arm, rows_s, width, seed, estimator, CFG.head_max_epochs, CFG.head_patience)
            CRITICS[(cell, arm, seed)] = critic
            inst = TUNING_INSTANCES[cell][0]
            counted = (2 * inst.m + 2 * (critic.head.dims[0] - hd.FEATURE_DIM) + hd.flops_amortised_per_interval(inst, ())
                       + sum(2 * a.shape[0] * a.shape[1] + a.shape[1] for a in critic.head.mlp.weights) + hd.TANH_FLOPS * sum(a.shape[1] for a in critic.head.mlp.weights[:-1])
                       + (4 * inst.m + 2 * inst.m + 1 + 3 * hd.LOG_FLOPS + 6 if critic.features != 'none' else 0)
                       + ((2 * (1 + inst.m) + sum(2 * a.shape[0] * a.shape[1] + a.shape[1] for a in critic.estimator.mlp.weights)
                           + hd.TANH_FLOPS * sum(a.shape[1] for a in critic.estimator.mlp.weights[:-1])) if critic.estimator is not None else 0))
            assert cr.flops_critic_per_interval(inst, critic) == counted
            est = critic.estimator
            TRAIN_ROWS.append({'cell': cell, 'arm': arm, 'seed': seed, 'head_width': width, 'head_parameters': critic.report['head_parameters'],
                               'estimator_parameters': critic.report['estimator_parameters'], 'head_epochs': critic.report['epochs_run'],
                               'head_val_rmse_log10': critic.report['val_rmse_log10'], 'target_std_log10': critic.report['target_std_log10'],
                               'head_val_spearman_within_state': critic.report['val_spearman_within_state'],
                               'estimator_epochs': est.report['epochs_run'] if est else None, 'estimator_hit_max_epochs': est.report['hit_max_epochs'] if est else None,
                               'estimator_val_error_true_costate': est.report['val_relative_error_against_true_costate'] if est else None,
                               'estimator_train_error_true_costate': est.report['train_relative_error_against_true_costate'] if est else None,
                               'estimator_floor_met': (est.report['val_relative_error_against_true_costate'] <= CFG.estimator_floor) if (est and not est.report['permuted_labels']) else None,
                               'flops_per_interval': cr.flops_critic_per_interval(inst, critic), 'steps_per_interval': cr.flops_critic_per_interval(inst, critic) / hd.flops_step(inst),
                               'lookup_price_scale': cr.lookup_price_scale(inst, critic),
                               'training_flops_steps': (critic.report['training_flops'] + (est.report['training_flops'] if est else 0)) / hd.flops_step(inst)})
        print(f'{cell} seed {seed}: {time.perf_counter() - started:.0f}s elapsed', flush=True)
TRAINING = pd.DataFrame(TRAIN_ROWS)
TRAINING.to_csv(PATHS['artifacts'] / 'training_summary.csv', index=False)
ESTIMATOR_OK = {cell: int(TRAINING[(TRAINING['cell'] == cell) & (TRAINING['arm'] == 'costate_critic')]['estimator_floor_met'].astype(bool).sum()) for cell in CFG.cells}
print(TRAINING[TRAINING['arm'].isin(['direct', 'costate_critic'])][['cell', 'arm', 'seed', 'head_width', 'head_val_rmse_log10', 'head_val_spearman_within_state', 'estimator_val_error_true_costate', 'steps_per_interval']].to_string(index=False))
print('estimator floor met in (of', len(CFG.seeds), 'seeds):', ESTIMATOR_OK)

In [ ]:
# ================================================================
# 5. Choose theta per (cell, arm, seed) on the tuning family at both prices, save the critics, then freeze
# ================================================================
STEP_DOUBLING = {'adjoint': lambda th: marking_policy('adjoint', th)}
REFERENCE_MAKERS = {'cheap': hd.cheap_policy, 'cheap_adjoint': hd.cheap_adjoint_policy, **STEP_DOUBLING}
SELECTED_THETA, TUNING_ROWS = {}, []
started = time.perf_counter()
for cell in CFG.cells:
    tuning = TUNING_INSTANCES[cell]
    for arm in CFG.reference_arms:
        options = hd.tune_theta(DOMAIN, tuning, REFERENCE_MAKERS[arm], CFG.thetas, CFG.gap_fractions, CFG.cap_in_finest_grids, CFG.max_passes, CFG.primary_method)
        SELECTED_THETA[(cell, arm, None, 'real')] = min(options, key=options.get)
        TUNING_ROWS.append({'cell': cell, 'arm': arm, 'seed': None, 'price': 'real', **{f'theta_{t:g}': options[t] for t in CFG.thetas}, 'selected': SELECTED_THETA[(cell, arm, None, 'real')]})
    for (c, arm, seed), critic in CRITICS.items():
        if c != cell:
            continue
        for price in (('lookup',) if arm == 'teacher_feature' else ('real', 'lookup')):
            options = cr.tune_critic_theta(DOMAIN, tuning, critic, CFG.thetas, price, targets_fn, cap_of, CFG.max_passes, CFG.primary_method)
            SELECTED_THETA[(cell, arm, seed, price)] = min(options, key=options.get)
            TUNING_ROWS.append({'cell': cell, 'arm': arm, 'seed': seed, 'price': price, **{f'theta_{t:g}': options[t] for t in CFG.thetas},
                                'selected': SELECTED_THETA[(cell, arm, seed, price)]})
    print(f'{cell}: {time.perf_counter() - started:.0f}s elapsed', flush=True)
TUNING = pd.DataFrame(TUNING_ROWS)
TUNING['selected_at_grid_edge'] = TUNING['selected'].isin([min(CFG.thetas), max(CFG.thetas)])
TUNING.to_csv(PATHS['artifacts'] / 'theta_tuning.csv', index=False)

CRITIC_HASHES = {}
for cell in CFG.cells:
    for arm in cr.ARMS:
        blob = {str(seed): CRITICS[(cell, arm, seed)].to_json() for seed in CFG.seeds}
        atomic_write_json(PATHS['artifacts'] / f'critics_{cell}_{arm}.json', blob)
        CRITIC_HASHES[f'{cell}/{arm}'] = sha256_json(blob)

FROZEN = {'config_sha256': CONFIG_HASH, 'train_manifest_sha256': sha256_json(TRAIN_MANIFEST), 'tuning_manifest_sha256': sha256_json(TUNING_MANIFEST),
          'widths': WIDTHS, 'estimator_floor_met_seeds': ESTIMATOR_OK, 'critic_sha256': CRITIC_HASHES,
          'selected_theta': {f'{c}/{a}/{s}/{p}': v for (c, a, s, p), v in SELECTED_THETA.items()},
          'rules': {'R0': 'costate_critic or direct beats uniform_pass at the real price', 'RC1': 'costate_critic beats direct (real price); RC1-L at the lookup price',
                    'RC2': 'the 95 % interval of log(costate_critic / direct) lies inside +-ln 1.05 at each of the three targets (RC2-L at the lookup price)',
                    'RC3': 'costate_critic beats costate_randomised (RC3-L at the lookup price)', 'RT-L': 'teacher_feature beats direct at the lookup price',
                    'estimator_floor': f'held-out relative error <= {CFG.estimator_floor} in at least {CFG.seeds_needed_for_estimator_floor} of {len(CFG.seeds)} seeds',
                    'beats': 'two-stage bootstrap 95 % interval of the mean log compute ratio below 0 at two adjacent targets (primary method log-log)'},
          'primary_method': CFG.primary_method, 'written_utc': datetime.datetime.now(datetime.UTC).isoformat()}
FROZEN['sha256'] = sha256_json({k: v for k, v in FROZEN.items() if k != 'written_utc'})
atomic_write_json(PATHS['reports'] / 'frozen_before_validation.json', FROZEN)
print('frozen', FROZEN['sha256'][:12], '| theta at the grid edge:', int(TUNING['selected_at_grid_edge'].sum()), 'of', len(TUNING))

In [ ]:
# ================================================================
# 6. Validation family: generated only now. Compute to reach each target at the real and the lookup price
# ================================================================
from adjointrwm.domains import evaluate_work_precision

VALIDATION_INSTANCES = {cell: hd.varying_goal_instances(cell, 'validation', CFG.validation_instances) for cell in CFG.cells}
instance_table(VALIDATION_INSTANCES, 'validation').to_parquet(PATHS['config'] / 'instances_validation.parquet', index=False)
VALIDATION_MANIFEST = manifest_of(VALIDATION_INSTANCES, 'validation', CFG.validation_instances)
atomic_write_json(PATHS['config'] / 'instances_manifest_validation.json', {**VALIDATION_MANIFEST, 'sha256': sha256_json(VALIDATION_MANIFEST)})

started = time.perf_counter()
FRAMES = {}
for cell in CFG.cells:
    instances = VALIDATION_INSTANCES[cell]
    refs = [uniform_pass_policy()] + [dataclasses.replace(REFERENCE_MAKERS[a](SELECTED_THETA[(cell, a, None, 'real')]), name=a) for a in CFG.reference_arms]
    parts = [evaluate_work_precision(DOMAIN, [inst], refs, targets_fn, compute_cap=cap_of(inst), max_steps=CFG.max_passes, random_draws=1,
                                     decision_scales=(1.0, 0.0), methods=CFG.methods) for inst in instances]
    ref_frame = pd.concat(parts, ignore_index=True)
    ref_frame = pd.concat([ref_frame[ref_frame['decision_scale'] == 1.0],
                           ref_frame[(ref_frame['decision_scale'] == 0.0) & (ref_frame['policy'] == 'adjoint')].assign(policy='adjoint_free', decision_scale=1.0)], ignore_index=True)
    critic_parts = []
    for (c, arm, seed), critic in CRITICS.items():
        if c != cell:
            continue
        for price in (('lookup',) if arm == 'teacher_feature' else ('real', 'lookup')):
            critic_parts.append(cr.evaluate_critic(DOMAIN, instances, critic, SELECTED_THETA[(cell, arm, seed, price)], price, targets_fn, cap_of, CFG.max_passes,
                                                   f'{arm}|s{seed}|{price}', methods=CFG.methods))
    frame = pd.concat([ref_frame] + critic_parts, ignore_index=True)
    frame.insert(0, 'cell', cell)
    FRAMES[cell] = frame
    print(f'{cell}: {time.perf_counter() - started:.0f}s elapsed', flush=True)
ALL = pd.concat(FRAMES.values(), ignore_index=True)
ALL.to_parquet(PATHS['artifacts'] / 'validation_work_precision.parquet', index=False)
print(len(ALL), 'rows written')

In [ ]:
# ================================================================
# 7. Frozen rules, tables, figure and report
# ================================================================
SEEDS = list(CFG.seeds)
PRICES = ('real', 'lookup')
COMPARISONS = [('costate_critic', 'direct'), ('costate_critic', 'costate_randomised'), ('teacher_feature', 'direct'), ('costate_critic', 'uniform_pass'),
               ('direct', 'uniform_pass'), ('costate_critic', 'cheap'), ('direct', 'cheap'), ('costate_critic', 'cheap_adjoint'), ('direct', 'cheap_adjoint'),
               ('teacher_feature', 'cheap_adjoint'), ('costate_randomised', 'direct'), ('costate_critic', 'adjoint_free'), ('direct', 'adjoint_free'),
               ('teacher_feature', 'uniform_pass'), ('cheap_adjoint', 'uniform_pass'), ('cheap', 'uniform_pass')]
ALL_ARMS = set(cr.ARMS)
INFERENCE, SUMMARY, RULES = [], {}, {}
for cell in CFG.cells:
    frame, names = FRAMES[cell], [i.name for i in VALIDATION_INSTANCES[cell]]
    cap = float(cap_of(VALIDATION_INSTANCES[cell][0]))
    SUMMARY[cell] = {}
    for method in CFG.methods:
        for price in PRICES:
            for target in TARGETS:
                def side(arm, price=price, target=target, method=method):
                    if arm in ALL_ARMS:
                        return cr.log_compute_matrix(frame, [f'{arm}|s{s}|{price}' for s in SEEDS], names, target, method, cap)
                    return cr.log_compute_matrix(frame, [arm], names, target, method, cap)[0][None, :]

                for a, b in COMPARISONS:
                    if price == 'real' and 'teacher_feature' in (a, b):
                        continue
                    diff = side(a) - side(b)
                    if diff.shape[0] == 1:
                        diff = np.repeat(diff, len(SEEDS), axis=0)
                    ci = cr.two_stage_bootstrap(diff, CFG.bootstrap_resamples, seed=0)
                    SUMMARY[cell].setdefault(method, {}).setdefault(price, {}).setdefault(target, {})[f'{a} / {b}'] = ci
                    INFERENCE.append({'cell': cell, 'method': method, 'price': price, 'target': target, 'a': a, 'b': b, **{k: v for k, v in ci.items() if k != 'per_seed'}})

    def flags_for(method):
        def row(price, a, b):
            return [SUMMARY[cell][method][price][t][f'{a} / {b}'] for t in TARGETS]

        def beats(price, a, b):
            return cr.two_adjacent([r['ci_high'] < 0 for r in row(price, a, b)])

        return {'R0': bool(beats('real', 'costate_critic', 'uniform_pass') or beats('real', 'direct', 'uniform_pass')),
                'RC1': bool(beats('real', 'costate_critic', 'direct')), 'RC1L': bool(beats('lookup', 'costate_critic', 'direct')),
                'RC2': bool(cr.within_margin(row('real', 'costate_critic', 'direct'), CFG.equivalence_margin)),
                'RC2L': bool(cr.within_margin(row('lookup', 'costate_critic', 'direct'), CFG.equivalence_margin)),
                'RC3': bool(beats('real', 'costate_critic', 'costate_randomised')), 'RC3L': bool(beats('lookup', 'costate_critic', 'costate_randomised')),
                'RTL': bool(beats('lookup', 'teacher_feature', 'direct')),
                'critic_beats_uniform_real': bool(beats('real', 'costate_critic', 'uniform_pass')), 'direct_beats_uniform_real': bool(beats('real', 'direct', 'uniform_pass')),
                'critic_beats_uniform_lookup': bool(beats('lookup', 'costate_critic', 'uniform_pass')), 'direct_beats_uniform_lookup': bool(beats('lookup', 'direct', 'uniform_pass')),
                'critic_beats_cheap_lookup': bool(beats('lookup', 'costate_critic', 'cheap')), 'direct_beats_cheap_lookup': bool(beats('lookup', 'direct', 'cheap')),
                'cheap_adjoint_beats_uniform': bool(beats('real', 'cheap_adjoint', 'uniform_pass'))}

    per_method = {m_: flags_for(m_) for m_ in CFG.methods}
    primary = per_method[CFG.primary_method]
    estimator_ok = ESTIMATOR_OK[cell] >= CFG.seeds_needed_for_estimator_floor
    RULES[cell] = {**primary, 'estimator_ok': bool(estimator_ok), 'estimator_floor_met_seeds': ESTIMATOR_OK[cell], 'saturated': bool(WIDTHS[cell]['direct_saturated']),
                   'exit_class': cr.exit_class({**primary, 'estimator_ok': estimator_ok, 'saturated': WIDTHS[cell]['direct_saturated']}),
                   'sensitivity_semilog': per_method['semilog'], 'sensitivity_staircase': per_method['staircase']}
    RULES[cell]['exit_class_semilog'] = cr.exit_class({**per_method['semilog'], 'estimator_ok': estimator_ok, 'saturated': WIDTHS[cell]['direct_saturated']})
    RULES[cell]['exit_class_staircase'] = cr.exit_class({**per_method['staircase'], 'estimator_ok': estimator_ok, 'saturated': WIDTHS[cell]['direct_saturated']})
    print(cell, '->', RULES[cell]['exit_class'], '| semilog:', RULES[cell]['exit_class_semilog'], '| staircase:', RULES[cell]['exit_class_staircase'])

INFERENCE_TABLE = pd.DataFrame(INFERENCE)
INFERENCE_TABLE.to_csv(PATHS['figures'] / 'comparisons.csv', index=False)
atomic_write_json(PATHS['artifacts'] / 'validation_summary.json', {'selected_theta': FROZEN['selected_theta'], 'comparisons': [f'{a} / {b}' for a, b in COMPARISONS],
                                                                   'methods': list(CFG.methods), 'primary_method': CFG.primary_method, 'summary': SUMMARY,
                                                                   'note': 'log compute ratios, two-stage bootstrap (seeds, then instances); teacher_feature and every lookup-price row are hypothetical'})

# retained fraction of the cheap -> cheap_adjoint headroom, and break-even instance counts (reported, not gated)
EXTRA, REF_RATIO = [], {}
for cell in CFG.cells:
    frame, names = FRAMES[cell], [i.name for i in VALIDATION_INSTANCES[cell]]
    cap = float(cap_of(VALIDATION_INSTANCES[cell][0]))
    inst = VALIDATION_INSTANCES[cell][0]
    for price in PRICES:
        for arm in cr.ARMS:
            if price == 'real' and arm == 'teacher_feature':
                continue
            for target in TARGETS:
                mat = cr.log_compute_matrix(frame, [f'{arm}|s{s}|{price}' for s in SEEDS], names, target, CFG.primary_method, cap)
                cheap, adj, uni = (cr.log_compute_matrix(frame, [n], names, target, CFG.primary_method, cap)[0] for n in ('cheap', 'cheap_adjoint', 'uniform_pass'))
                headroom = cheap.mean() - adj.mean()
                REF_RATIO[(cell, target)] = {'cheap': float(np.exp(cheap.mean() - uni.mean())), 'cheap_adjoint': float(np.exp(adj.mean() - uni.mean()))}
                kept = [(cheap.mean() - mat[k].mean()) / headroom if headroom > 0 else None for k in range(len(SEEDS))]
                train_steps = TRAINING[(TRAINING['cell'] == cell) & (TRAINING['arm'] == arm)]['training_flops_steps'].to_numpy()
                saved = np.exp(uni)[None, :] - np.exp(mat)
                EXTRA.append({'cell': cell, 'price': price, 'arm': arm, 'target': target,
                              'retained_cheap_to_cheap_adjoint_mean': (float(np.mean(kept)) if headroom > 0 else None), 'retained_min': (float(np.min(kept)) if headroom > 0 else None),
                              'retained_max': (float(np.max(kept)) if headroom > 0 else None),
                              'geometric_mean_compute': float(np.exp(mat.mean())), 'ratio_to_uniform': float(np.exp(mat.mean() - uni.mean())),
                              'break_even_instances_mean_over_seeds': (float(np.mean([train_steps[k] / saved[k].mean() for k in range(len(SEEDS)) if saved[k].mean() > 0]))
                                                                       if (saved.mean(axis=1) > 0).any() else None)})
EXTRA_TABLE = pd.DataFrame(EXTRA)
EXTRA_TABLE.to_csv(PATHS['figures'] / 'retained_and_break_even.csv', index=False)

import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(CFG.cells), len(TARGETS), figsize=(4.6 * len(TARGETS), 3.6 * len(CFG.cells)), sharex=False, squeeze=False)
for r, cell in enumerate(CFG.cells):
    for c, target in enumerate(TARGETS):
        ax = axes[r][c]
        sub = EXTRA_TABLE[(EXTRA_TABLE['cell'] == cell) & (EXTRA_TABLE['target'] == target)]
        labels = [f'{arm} ({price})' for price in PRICES for arm in cr.ARMS if not (price == 'real' and arm == 'teacher_feature')]
        values = [float(sub[(sub['arm'] == arm) & (sub['price'] == price)]['ratio_to_uniform'].iloc[0]) for price in PRICES for arm in cr.ARMS if not (price == 'real' and arm == 'teacher_feature')]
        ax.barh(range(len(values)), values, color=['C0' if 'real' in lab else 'C1' for lab in labels])
        ax.axvline(1.0, color='grey', lw=1)
        for ref, colour in (('cheap', 'C2'), ('cheap_adjoint', 'C3')):
            ax.axvline(REF_RATIO[(cell, target)][ref], color=colour, lw=1, ls='--', label=ref)
        ax.set_xscale('log'); ax.set_yticks(range(len(values))); ax.set_yticklabels(labels, fontsize=7)
        ax.set_title(f'{cell}, {target[3:]} of the gap left', fontsize=9)
        ax.set_xlabel('compute vs uniform_pass (below 1: cheaper)', fontsize=8)
axes[0][0].legend(fontsize=7)
fig.suptitle(f'{RUN_ID}: validation, geometric-mean compute ratio over seeds (orange: hypothetical lookup price)', fontsize=9)
fig.savefig(PATHS['figures'] / 'compute_ratios.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

CELLS = pd.DataFrame([{'cell': cell, 'm': VALIDATION_INSTANCES[cell][0].m, 'head_width': WIDTHS[cell]['width'], 'direct_saturated': WIDTHS[cell]['direct_saturated'],
                       'estimator_floor_met_seeds': ESTIMATOR_OK[cell], **{k: v for k, v in RULES[cell].items() if not isinstance(v, dict)}} for cell in CFG.cells])
CELLS.to_csv(PATHS['figures'] / 'phase_table.csv', index=False)
REPORT = {
    'run_id': RUN_ID, 'milestone': 'D4-1 (exploratory design study)', 'config_hash': CONFIG_HASH, 'frozen_sha256': FROZEN['sha256'], 'repo_commit': REPO_COMMIT,
    'correctness': CORRECTNESS, 'widths': WIDTHS, 'estimator_floor_met_seeds': ESTIMATOR_OK, 'rules': RULES,
    'theta_selected_at_grid_edge': int(TUNING['selected_at_grid_edge'].sum()), 'theta_tuned': int(len(TUNING)),
    'test_family_read': False,
    'claim_boundary': ('Exploratory design study on a train, a tuning and a validation family of the analytic D4 benchmark with a random unit goal per instance; the test family was '
                       'not read. FLOPs are counted from the code path, not measured. The lookup price and the teacher_feature arm are hypothetical; the validation instances were read '
                       'before for the non-learned arms of D4-3. No statement about real data or about H2 there follows.'),
}
atomic_write_json(PATHS['reports'] / 'acceptance_report.json', REPORT)
lines = [f'# D4-1 `{RUN_ID}` (exploratory)', '', f"Correctness: **{'PASS' if CORRECTNESS['passed'] else 'FAIL'}**", '',
         '| Cell | Head width | D saturated | Estimator floor met (seeds) | R0 | RC1 | RC1-L | RC2-L | RC3 | RC3-L | RT-L | Exit class |', '|---|---:|---|---:|---|---|---|---|---|---|---|---|']
for _, r in CELLS.iterrows():
    lines.append(f"| {r['cell']} | {r['head_width']} | {r['direct_saturated']} | {r['estimator_floor_met_seeds']} | {r['R0']} | {r['RC1']} | {r['RC1L']} | {r['RC2L']} | {r['RC3']} | {r['RC3L']} | {r['RTL']} | {r['exit_class']} |")
lines += ['', REPORT['claim_boundary']]
atomic_write_text(PATHS['reports'] / 'run_summary.md', '\n'.join(lines) + '\n')
atomic_write_text(RUN_DIR / 'COMPLETE', datetime.datetime.now(datetime.UTC).isoformat() + '\n')
print('\n'.join(lines))
print('\nRun directory:', RUN_DIR)

## Reading the result

The exit class per cell is the plan's §7 table applied to the frozen rules; read `reports/acceptance_report.json` (and its semilog and staircase sensitivities) first, then `figures/comparisons.csv` for the intervals behind every rule.

| Rule outcome | Meaning |
|---|---|
| R0, RC1 and RC3 hold at the real price, estimator floor met | A learned co-state critic beats the information-equivalent direct critic at its real price in this analytic domain; next a confirmatory plan on the test family |
| RC1-L and RC3-L hold but not at the real price | The ranking is better, the network's price eats it |
| RT-L holds, the estimator floor fails or RC1-L fails | The exact co-state helps a learned head; a learned generic estimator does not deliver it |
| RC2-L holds and RT-L does not | Direct utility sufficient in this domain |
| none of these | Inconclusive; read the intervals and the width probe |

Read `artifacts/training_summary.csv` (estimator error against the exact co-state, head fit, prices per interval) before any statement about why. Every lookup-price row and the `teacher_feature` arm are hypothetical and say nothing about a deployable critic.